In [145]:
import pandas as pd 
import json
df = pd.read_json('data.json')
df_alerts = pd.json_normalize(df["alerts"].explode().dropna())

In [146]:
# Remove contacts that do not contain alerts
df_cleanup = df[df.alerts.str.len() > 0]

# Convert alert data from json
df_cleanup = df_cleanup.explode('alerts').reset_index(drop=True)
df_cleanup_alerts = pd.json_normalize(df_cleanup.alerts)

# Replace alert column with data per every field
df_cleanup = df_cleanup.drop('alerts', axis=1)
df_cleanup = pd.concat([df_cleanup, df_cleanup_alerts], axis=1)

In [147]:
# Convert Datetime columns to Datetime type
df_cleanup.contactBeginTimestamp = pd.to_datetime(df_cleanup.contactBeginTimestamp, unit='s')
df_cleanup.contactEndTimestamp = pd.to_datetime(df_cleanup.contactEndTimestamp, unit='s')
df_cleanup.errorTime = pd.to_datetime(df_cleanup.errorTime, unit='ms')

# Add contactTime
df_cleanup['contactTime'] = df_cleanup.contactEndTimestamp - df_cleanup.contactBeginTimestamp

In [148]:
df_cleanup.to_json(
    "loisel-challenge/public/data_processed.json", orient="records", date_format="iso"
)